# Consommation énergétique finale mondiale : historique vs scénarios

Données : `../data/preproc/world_total_final_consumption.xlsx` (historical/current/stated/netzero si dispo).
Objectifs :
- Courbes par scénario (pointillés >2025).
- Écarts 2035/2050 et CAGR.
- Part de l’électricité dans la consommation finale (si colonnes disponibles en support), pour relier à l’électrification.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9,5)
root_pre = Path('..') / 'data' / 'preproc'


In [2]:
cons = pd.read_excel(root_pre / 'world_total_final_consumption.xlsx')
cons = cons.rename(columns={'Total_final_consumption_TWh':'value','Scenario':'scenario','Year':'Year'})
cons['line_dash'] = np.where(cons['Year']>2025,'dash','solid')
fig = px.line(cons, x='Year', y='value', color='scenario', line_dash='line_dash', markers=True,
              title='Consommation énergétique finale mondiale (pointillés >2025)')
fig.show()


In [3]:
# Écarts 2035/2050
summary = cons[cons['Year'].isin([2035,2050])].pivot(index='Year', columns='scenario', values='value')
summary


scenario,CURRENT,NZE,STATED
Year,,,
2035,145278.94,113334.24,140001.12
2050,166390.22,97778.56,150278.98


In [4]:
# CAGR 2024->2035/2050
cagr_rows=[]
for sc in cons['scenario'].unique():
    sc_data = cons[cons['scenario']==sc]
    if 2024 in sc_data['Year'].values:
        base = sc_data.loc[sc_data['Year']==2024,'value'].values[0]
        for target in [2035,2050]:
            if target in sc_data['Year'].values and base>0:
                val = sc_data.loc[sc_data['Year']==target,'value'].values[0]
                yrs = target-2024
                cagr_rows.append({'scenario':sc,'target_year':target,'CAGR':(val/base)**(1/yrs)-1})
pd.DataFrame(cagr_rows)


,scenario,target_year,CAGR
0,CURRENT,2035,0.013148
1,CURRENT,2050,0.010803
2,STATED,2035,0.009746
3,STATED,2050,0.006851
4,NZE,2035,-0.009466
5,NZE,2050,-0.009655


## Électricité vs consommation finale (si souhaité)


In [5]:
try:
    elec = pd.read_excel(root_pre / 'world_total_electricity_generation.xlsx')
    elec = elec.rename(columns={'Total_generation_TWh':'value','Scenario':'scenario','Year':'Year'})
    elec_sum = elec.groupby('Year')['value'].mean().reset_index().rename(columns={'value':'elec_TWh'})
    cons_merge = cons.merge(elec_sum, on='Year', how='left')
    cons_merge['elec_share_%'] = cons_merge['elec_TWh']/cons_merge['value']*100
    fig = px.line(cons_merge, x='Year', y='elec_share_%', markers=True, title='Part de l’électricité dans la consommation finale (approx.)')
    fig.show()
except FileNotFoundError:
    pass


## Lecture rapide
- Les courbes montrent la tension entre scénarios : current vs stated/netzero.
- Les écarts 2035/2050 et CAGR quantifient la trajectoire de demande.
- La part de l’électricité (si calculée) donne une idée du niveau d’électrification visé.
